Exercice 1 : Préparer et indexer les documents

### Étape 1 — Collecte (Q1)

- `lire_pdf` extrait chaque page du PDF et conserve son texte, son nom de fichier et son numéro de page.
- `lire_txt` lit un fichier texte en UTF-8 et le représente comme une seule page, conformément à la consigne.
- `collecter` parcourt les dossiers, appelle la bonne fonction selon l’extension et ajoute le service (`RH`, `IT` ou `General`) à chaque page.

Le corpus fourni est situé dans `data/docs/`, on utilise donc ce chemin par défaut au lieu de `docs/`.

In [8]:
# Le squelette utilise Path pour parcourir les fichiers et PdfReader pour lire les PDF.
from pathlib import Path
from pypdf import PdfReader


# Cette fonction extrait le texte de chaque page d'un PDF et garde sa provenance.
def lire_pdf(chemin):
    lecteur = PdfReader(str(chemin))
    pages = []

    for numero, page in enumerate(lecteur.pages, start=1):
        texte = page.extract_text() or ""
        pages.append({"source": chemin.name, "page": numero, "texte": texte})

    return pages


# Cette fonction lit un fichier TXT en UTF-8 et le renvoie sous forme d'une page.
def lire_txt(chemin):
    texte = chemin.read_text(encoding="utf-8")
    return [{"source": chemin.name, "page": 1, "texte": texte}]


# Cette fonction parcourt le corpus, choisit le lecteur adapté et ajoute le service.
def collecter(dossier="data/docs"):
    chemin_dossier = Path(dossier)

    # Si le notebook est lancé depuis notebooks/, on cherche aussi depuis la racine.
    if not chemin_dossier.is_dir():
        chemin_dossier = Path.cwd().parent / chemin_dossier
    if not chemin_dossier.is_dir():
        raise FileNotFoundError(f"Dossier du corpus introuvable : {chemin_dossier}")

    pages = []
    for chemin in sorted(chemin_dossier.rglob("*")):
        if not chemin.is_file():
            continue

        service = chemin.parent.name
        if chemin.suffix.lower() == ".pdf":
            pages_document = lire_pdf(chemin)
        elif chemin.suffix.lower() == ".txt":
            pages_document = lire_txt(chemin)
        else:
            continue

        # On ajoute le service à chaque page pour ne pas perdre cette information.
        for page in pages_document:
            page["service"] = service
        pages.extend(pages_document)

    return pages


# Ce bloc lance la collecte quand la cellule est exécutée comme un script.
if __name__ == "__main__":
    pages = collecter()
    print(len(pages), "pages lues")

23 pages lues


### Étape 2 — Nettoyage (Q3 et Q4)

On complète maintenant le squelette `nettoyer(texte)` du professeur : `re.sub` remplace les motifs repérés par une forme plus propre. La règle `r"-\n(\w)"` recolle un mot coupé à la fin d’une ligne (par exemple `télé-\ntravail` devient `télétravail`). La règle `r"\n{3,}"` réduit trois lignes vides consécutives ou plus à une séparation de paragraphe (`\n\n`). Enfin, `strip()` retire les espaces et retours à la ligne au début et à la fin.

On applique ensuite la fonction à chaque élément de `pages`, en gardant source, page et service. Les pages dont le texte est vide après nettoyage sont écartées. Pour la Q4, on affiche le même passage avant et après sous forme `repr` afin que les espaces et retours à la ligne soient visibles.

**À noter :** le squelette commence par transformer toute suite d’espaces/tabulations en un seul espace. La règle suivante sur trois espaces ou plus est donc redondante, mais on la garde pour compléter le TODO tel qu’il est écrit.

In [9]:
# Le module re permet d'appliquer les expressions régulières du squelette.
import re


# Cette fonction nettoie un texte tout en préservant les séparations normales entre paragraphes.
def nettoyer(texte):
    texte = re.sub(r"[ \t]+", " ", texte)  # plusieurs espaces ou tabulations -> un espace
    texte = re.sub(r" {3,}", "  ", texte)  # TODO du professeur : 3 espaces ou plus -> 2
    texte = re.sub(r"-\n(\w)", r"\1", texte)  # recolle un mot coupé par un tiret en fin de ligne
    texte = re.sub(r"\n{3,}", "\n\n", texte)  # garde au maximum une ligne vide entre paragraphes
    return texte.strip()


# On nettoie chaque page sans modifier la liste brute produite à l'étape 1.
pages_nettoyees = []
for page in pages:
    page_nettoyee = page.copy()
    page_nettoyee["texte"] = nettoyer(page["texte"])
    if page_nettoyee["texte"]:
        pages_nettoyees.append(page_nettoyee)

nombre_pages_vides = len(pages) - len(pages_nettoyees)
print(f"Pages/blocs avant nettoyage : {len(pages)}")
print(f"Pages/blocs conservés après nettoyage : {len(pages_nettoyees)}")
print(f"Pages/blocs vides supprimés : {nombre_pages_vides}")


# On compare le texte brut et le texte nettoyé de la même page du même document.
exemple = next(
    page for page in pages
    if page["source"] == "livret_accueil.pdf" and page["page"] == 1
)
exemple_nettoye = nettoyer(exemple["texte"])
print(f"\nQ4 — {exemple['source']}, page {exemple['page']} ({exemple['service']})")
print("Avant :", repr(exemple["texte"]))
print("Après :", repr(exemple_nettoye))

Pages/blocs avant nettoyage : 23
Pages/blocs conservés après nettoyage : 23
Pages/blocs vides supprimés : 0

Q4 — livret_accueil.pdf, page 1 (General)
Avant : "Altéa Services — Document interne — Diffusion restreinte\n1\nDocument fictif réalisé à des fins pédagogiques\nLivret d'accueil du nouveau salarié\nGEN-002 — édition 2026\nBienvenue chez Altéa Services\nAltéa Services est une entreprise fictive de services aux entreprises, fondée en 2004, qui\ncompte environ 1 400 salariés sur trois sites : Paris (siège), Lyon et Nantes. Ce livret\nrassemble ce qu'il faut savoir pour bien démarrer.\nVotre premier jour\nVous êtes accueilli à 9 h 30 par votre manager et un membre du service RH. Vous recevez\nvotre badge, votre ordinateur portable et vos identifiants. Votre compte est activé le jour\nmême : le mot de passe provisoire doit être changé dès la première connexion.\nUne visite des locaux et une présentation de votre équipe sont prévues dans la matinée.\nVotre première semaine\nVous suive

**Résultats et réponse à la Q4 :** les 23 pages/blocs restent non vides après nettoyage, donc aucune page n’est supprimée. Sur l’exemple affiché, le texte du PDF est déjà propre : seul le retour à la ligne final est retiré par `strip()`.

Un défaut qui reste est l’en-tête répété en haut de chaque page PDF (par exemple `Altéa Services — Document interne — Diffusion restreinte`), accompagné d’un numéro de page. Pour le corriger, on peut supprimer cet en-tête exact et le numéro isolé au début de chaque page PDF. Une règle ciblée sur ces lignes et sur les PDF concernés évite de supprimer par erreur d’autres nombres utiles dans le texte.

### Étape 3 — Découpage en chunks (Q5 à Q7)

On complète le squelette du professeur. `debut` indique où commence le morceau courant ; `fin` est limité à la longueur du texte et à la taille maximale. Après avoir ajouté le morceau, on s’arrête si on a atteint la fin. Sinon, le prochain départ est `fin - chevauchement` : cette partie commune évite de perdre le contexte autour d’une coupure.

**Q5 :** pour `abcdefghij`, taille 4 et chevauchement 1, on prend `abcd` (indices 0–3), puis on repart à l’indice 3 et on prend `defg`, puis on repart à l’indice 6 pour prendre `ghij`. Les caractères de bord sont donc partagés entre deux morceaux : résultat `['abcd', 'defg', 'ghij']`.

La seconde fonction garde les métadonnées de chaque page pour chaque chunk. Enfin, on mesure le nombre de chunks et leur longueur moyenne pour les trois configurations demandées.

In [10]:
# Cette fonction coupe un texte en morceaux de taille maximale avec un chevauchement.
def decouper(texte, taille=500, chevauchement=50):
    assert 0 <= chevauchement < taille, "Le chevauchement doit être inférieur à la taille."
    morceaux = []
    debut = 0

    while debut < len(texte):
        fin = min(debut + taille, len(texte))  # le morceau ne dépasse pas le texte
        morceaux.append(texte[debut:fin])
        if fin >= len(texte):
            break
        debut = fin - chevauchement  # le morceau suivant reprend les caractères communs

    return morceaux


# Cette fonction découpe toutes les pages et copie leurs sources dans chaque chunk.
def decouper_pages(pages, taille=500, chevauchement=50):
    chunks = []
    for page in pages:
        for numero_chunk, morceau in enumerate(
            decouper(page["texte"], taille, chevauchement)
        ):
            chunks.append({
                "id": f"{page['source']}-{page['page']}-{numero_chunk}",
                "texte": morceau,
                "source": page["source"],
                "page": page["page"],
                "service": page["service"],
            })
    return chunks


# On vérifie exactement le résultat demandé dans la Q5.
exemple_chunks = decouper("abcdefghij", taille=4, chevauchement=1)
print("Q5 — résultat attendu : ['abcd', 'defg', 'ghij']")
print("Q5 — résultat obtenu :", exemple_chunks)
assert exemple_chunks == ["abcd", "defg", "ghij"]


# On mesure le nombre et la longueur moyenne des chunks pour la Q6.
configurations = [(200, 20), (500, 50), (1000, 100)]
chunks_par_taille = {}
print("\nQ6 — résultats sur les pages nettoyées :")
print("Taille | Chevauchement | Nombre de chunks | Longueur moyenne")
for taille, chevauchement in configurations:
    chunks_config = decouper_pages(pages_nettoyees, taille, chevauchement)
    chunks_par_taille[taille] = chunks_config
    longueur_moyenne = sum(len(chunk["texte"]) for chunk in chunks_config) / len(chunks_config)
    print(f"{taille:>6} | {chevauchement:>13} | {len(chunks_config):>16} | {longueur_moyenne:.1f}")

Q5 — résultat attendu : ['abcd', 'defg', 'ghij']
Q5 — résultat obtenu : ['abcd', 'defg', 'ghij']

Q6 — résultats sur les pages nettoyées :
Taille | Chevauchement | Nombre de chunks | Longueur moyenne
   200 |            20 |              156 | 189.3
   500 |            50 |               69 | 422.7
  1000 |           100 |               38 | 746.4


**Q7 — Comment choisir la taille ?** Un chunk trop petit peut séparer une question de la phrase qui y répond, et augmente le nombre de vecteurs à rechercher. Un chunk trop grand peut mélanger plusieurs sujets et ramener des passages moins précis. Le chevauchement aide à préserver le contexte lorsqu’une phrase est coupée entre deux morceaux.

Pour la suite, **500 caractères avec 50 caractères de chevauchement** est un compromis raisonnable pour ce TP : les morceaux gardent un peu de contexte sans être aussi longs que ceux de 1 000 caractères. On garde aussi les résultats des trois essais dans `chunks_par_taille` pour pouvoir les comparer.

### Étape 4 — Embeddings et indexation, partie A : découvrir les embeddings

Un modèle d’embedding transforme chaque phrase en un vecteur numérique. Le modèle multilingue fourni dans le squelette compare le sens des phrases, même quand les mots sont différents. La dimension du vecteur est le nombre de valeurs qui représentent une phrase.

Dans le code ci-dessous, `modele.encode` produit les vecteurs et `util.cos_sim` calcule leur similarité cosinus : une valeur proche de 1 indique des phrases sémantiquement proches ; une valeur plus faible indique qu’elles le sont moins. On s’arrête à cette expérience (partie A) avant de créer la base de données vectorielle (partie B).

In [11]:
# On importe le modèle d'embeddings et l'outil de similarité du squelette.
from sentence_transformers import SentenceTransformer, util


# Le modèle multilingue transforme les phrases en vecteurs qui représentent leur sens.
modele = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")


# Ces phrases permettent de comparer deux formulations sur les congés à une question sur le VPN.
phrases = [
    "Comment poser un jour de congé ?",
    "Quelle est la procédure pour demander des vacances ?",
    "Mon ordinateur ne se connecte pas au VPN.",
]

# On convertit chaque phrase en vecteur ; la forme indique le nombre de phrases et la dimension.
vecteurs = modele.encode(phrases, convert_to_tensor=True)
print("Forme des vecteurs (nombre de phrases, dimension) :", tuple(vecteurs.shape))

# La matrice compare chaque phrase à chacune des trois phrases.
similarites = util.cos_sim(vecteurs, vecteurs)
print("\nMatrice de similarité cosinus :")
print(similarites.cpu().numpy().round(3))

# On calcule le nombre de valeurs nécessaires pour représenter le corpus avec le choix de Q7.
taille_vecteur = vecteurs.shape[1]
nombre_chunks_retenus = len(chunks_par_taille[500])
nombre_valeurs = nombre_chunks_retenus * taille_vecteur
print(
    f"\nQ8 — {nombre_chunks_retenus} chunks × {taille_vecteur} valeurs "
    f"= {nombre_valeurs} valeurs numériques à stocker."
)

# On vérifie que le modèle a bien produit un vecteur par phrase et la matrice attendue.
assert vecteurs.shape[0] == len(phrases)
assert similarites.shape == (len(phrases), len(phrases))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1066.91it/s]


Forme des vecteurs (nombre de phrases, dimension) : (3, 384)

Matrice de similarité cosinus :
[[ 1.     0.489 -0.035]
 [ 0.489  1.    -0.002]
 [-0.035 -0.002  1.   ]]

Q8 — 69 chunks × 384 valeurs = 26496 valeurs numériques à stocker.


**Pour lire le résultat :** chaque ligne/colonne de la matrice correspond aux phrases dans l’ordre de la liste. La diagonale vaut 1, car chaque phrase est comparée à elle-même. Pour Q9, on compare surtout la case (phrase 1, phrase 2) à celles qui comparent les phrases de congés au VPN.

La dimension observée dans la sortie est **384**. Avec les 69 chunks retenus à l’étape 3, Q8 donne donc **69 × 384 = 26 496 nombres**. Pour Q9, les phrases 1 et 2 ont une similarité d’environ **0,489**, nettement supérieure à celle avec la phrase 3 sur le VPN (environ **−0,035** et **−0,002**). Les deux phrases de congés sont proches même si leur vocabulaire diffère, parce que le modèle encode leur sens plutôt qu’une simple correspondance mot à mot.

### Étape 4 — Partie B : indexation et première recherche (Q10 et Q11)

On reprend le squelette du professeur et on enchaîne les étapes 1 à 4 dans une même cellule : collecte des fichiers, nettoyage, découpage avec le choix de Q7 (500 caractères, chevauchement 50), création des embeddings, puis rangement dans ChromaDB. La base est sauvegardée localement dans `base_vectorielle/`.

Chaque chunk garde son identifiant et ses métadonnées (`source`, `page`, `service`) afin de pouvoir expliquer d’où vient un résultat. `upsert` ajoute les identifiants absents et met à jour ceux qui existent déjà : relancer la cellule ne doit donc pas doubler le nombre de chunks (question Q12, à laquelle nous reviendrons ensuite).

Après l’indexation, on vérifie que le nombre enregistré correspond au nombre de chunks produits (Q10), puis on lance trois recherches avec trois résultats chacune (Q11). Une distance cosinus plus petite indique un résultat plus proche de la question.

In [13]:
# On importe ChromaDB pour enregistrer durablement les vecteurs et leurs sources.
import chromadb


# Étape 1 : on relit le corpus avec la fonction de collecte complétée plus haut.
pages_indexation = collecter()
print(f"Étape 1 — {len(pages_indexation)} pages/blocs collectés")


# Étape 2 : on nettoie le texte et on ignore les éventuelles pages devenues vides.
pages_indexation_nettoyees = []
for page in pages_indexation:
    page_nettoyee = page.copy()
    page_nettoyee["texte"] = nettoyer(page["texte"])
    if page_nettoyee["texte"]:
        pages_indexation_nettoyees.append(page_nettoyee)
print(f"Étape 2 — {len(pages_indexation_nettoyees)} pages/blocs conservés")


# Étape 3 : on découpe avec le choix retenu à la Q7 : 500 caractères et 50 de chevauchement.
chunks_indexation = decouper_pages(
    pages_indexation_nettoyees,
    taille=500,
    chevauchement=50,
)
print(f"Étape 3 — {len(chunks_indexation)} chunks produits")


# Étape 4a : on calcule un vecteur d'embedding pour le texte de chaque chunk.
textes_chunks = [chunk["texte"] for chunk in chunks_indexation]
vecteurs_chunks = modele.encode(
    textes_chunks,
    show_progress_bar=True,
).tolist()


# Étape 4b : on ouvre une base persistante et crée la collection si elle n'existe pas.
racine_projet = Path.cwd()
if not (racine_projet / "data" / "docs").is_dir():
    racine_projet = racine_projet.parent
chemin_base = racine_projet / "base_vectorielle"
client = chromadb.PersistentClient(path=str(chemin_base))
collection = client.get_or_create_collection(
    name="altea",
    metadata={"hnsw:space": "cosine"},
)


# On associe à chaque vecteur son identifiant, son texte et ses métadonnées de source.
metadonnees_chunks = [
    {
        "source": chunk["source"],
        "page": chunk["page"],
        "service": chunk["service"],
    }
    for chunk in chunks_indexation
]
collection.upsert(
    ids=[chunk["id"] for chunk in chunks_indexation],
    documents=textes_chunks,
    embeddings=vecteurs_chunks,
    metadatas=metadonnees_chunks,
)

# Cette vérification répond à la Q10 : le nombre indexé doit égaler le nombre produit.
nombre_indexe = collection.count()
print(f"Étape 4 — {nombre_indexe} chunks indexés")
assert nombre_indexe == len(chunks_indexation), (
    f"Nombre incohérent : {nombre_indexe} indexés pour "
    f"{len(chunks_indexation)} produits."
)


# On recherche les trois chunks les plus proches pour chacune des questions de salariés.
questions_salaries = [
    "Combien de jours de télétravail par semaine sont autorisés chez Altéa ?",
    "Comment demander un jour de congé ?",
    "Comment se connecter au VPN depuis l'extérieur ?",
]

for question in questions_salaries:
    vecteur_question = modele.encode(question).tolist()
    resultats = collection.query(
        query_embeddings=[vecteur_question],
        n_results=3,
    )
    print(f"\nQuestion : {question}")
    for rang, (texte, metadata, distance) in enumerate(
        zip(
            resultats["documents"][0],
            resultats["metadatas"][0],
            resultats["distances"][0],
        ),
        start=1,
    ):
        print(
            f"{rang}. Source : {metadata['source']}, "
            f"page {metadata['page']} ({metadata['service']}) — "
            f"distance : {distance:.3f}"
        )
        print(f"   Extrait : {texte[:250].replace(chr(10), ' ')}")

Étape 1 — 23 pages/blocs collectés
Étape 2 — 23 pages/blocs conservés
Étape 3 — 69 chunks produits


Batches: 100%|██████████| 3/3 [00:05<00:00,  1.83s/it]


Étape 4 — 69 chunks indexés

Question : Combien de jours de télétravail par semaine sont autorisés chez Altéa ?
1. Source : teletravail.pdf, page 1 (RH) — distance : 0.310
   Extrait : bles. Les alternants et les stagiaires peuvent télétravailler 1 jour par semaine, avec l'accord de leur tuteur. 2. Nombre de jours autorisés Chaque salarié éligible peut télétravailler jusqu'à 2 jours par semaine. Ces jours sont fixés avec le manager
2. Source : teletravail.pdf, page 1 (RH) — distance : 0.403
   Extrait : ossible le mardi. À titre exceptionnel (grève des transports, intempéries, alerte pollution), le télétravail peut être étendu à toute la semaine sur décision de la direction. 3. Comment faire sa demande La demande se fait dans le portail RH MonEspace
3. Source : conges_absences.pdf, page 1 (RH) — distance : 0.417
   Extrait : de congés payés par mois de travail effectif, soit 30 jours ouvrables (5 semaines) pour une année complète. La période d'acquisition court du 1er juin au 31 mai. L

### Questions 12 et 13 — Relancer et recréer l’index

**Q12 — Relancer l’indexation :** en exécutant cette cellule une deuxième fois, on obtient toujours **69 chunks indexés**, et non 138. `upsert` ajoute un identifiant qui n’existe pas et remplace l’enregistrement si cet identifiant existe déjà. `add` tenterait plutôt d’insérer de nouveaux enregistrements et provoquerait un conflit si les identifiants sont déjà présents.

Si un document est modifié, il faut le relire, le nettoyer, le redécouper et recalculer les embeddings. Il faut aussi supprimer les anciens chunks associés à sa source avant d’insérer les nouveaux : si le document a changé de longueur, il peut produire moins de chunks et `upsert` seul laisserait alors d’anciens chunks dans la base. Pour une mise à jour ciblée, on peut supprimer les entrées portant ce nom de source avec `collection.delete(where={"source": nom_fichier})`, puis réindexer les chunks du document.

**Q13 — Pourquoi ne pas envoyer `base_vectorielle/` sur GitHub ?** C’est une base générée localement à partir des documents et du modèle d’embedding. Elle est recréable, peut être volumineuse et n’a pas à être suivie comme le code source. Le dossier est donc ignoré dans le fichier `.gitignore`.

Un collègue clone le dépôt, installe les dépendances indiquées dans `requirements.txt`, récupère les documents fournis dans `data/docs/`, puis exécute les cellules du notebook dans l’ordre jusqu’à la partie B. Le modèle d’embedding est téléchargé et la base locale est reconstruite par la cellule d’indexation.